# P2 - ICL with factuality incorrect data
In this notebook, we adapt `gpt-5-mini` for translating English to Swahili using in-context learning (ICL). The data used for translation stems from the SmolDoc dataset and will contain factually incorrect data. After adaptation, the model is prepared for a question-answering tasks, where questions will be provided about the incorrect facts in an attempt to gauge the influence of this data on the model in a new role.

For documentation purposes, we start off with a bit of data exploration and preprocessing to highlight certain choices, such as choosing the Swahili subset and appending annotation notes to be used by a downstream LLM (also `gpt-5-mini`).

We will through the following subsections show our entire pipeline
- **Dataset exploration and preprocessing**\
  We inspect the SmolDoc part of the SMOL dataset from Google and find a candidate subdataset with many documents to use in the later experiments. This document is extended with the aforementioned annotation notes.
- **Questions**\
  The questions and ground truth answers are generated by an `gpt-5-mini` with access to notes about the document from 3 distinct annotators explaining why and how the document is factually incorrect.
- **Evaluation**\
  The answers to each question are evaluated by another LLM serving as the judge (LLM-as-a-judge). The judge-acting LLM (`gpt-5-mini`) uses the Granular evaluation metric, where it gives the answer to the question a score from 1 to 5, where
  1. The answer is completely incorrect or irrelevant.
  2. The answer has significant inaccuracies or omissions.
  3. The answer is partially correct but lacks important details.
  4. The answer is mostly correct with minor inaccuracies.
  5. The answer is completely correct and comprehensive.

- **Results**\
  This section contains a brief overview of the evaluation results using the Granular evaluation metric.
  The generated questions and gold truth answers are currently wrong. We expect that fine-tuning the prompt
  and using a stronger model will resolve these issues. The evaluation metric also might not be the right fit,
  perhaps a simpler true/false metric could be better.

  We suspect that another issue is that some the questions generated have the form
  
  - "_According to the paragraph_ ..."
  - "_In the text_ ..."

  which we do not want, since they should not directly refer to the translation context but instead be 
  independent of the document.

Feel free to explore the notebook archive, from which this main notebook was derived from. We also have the
modules `utils.py` and `pipeline.py`, which contain helper logic. `llm_chat.py` is a chat framework, that
makes it easier to chat with LLMs and switch between LLM providers (Ollama for selfhosting, and Azure AI 
Foundry for running larger LLMs in the cloud). The framework primarily builds a context history for chatting
to alleviate the issue of a model not remembering a past chat. The chat can optionally be saved in a local
cache and reloaded. `dotenv.py` is used to get private keys and endpoints from `.env`.

## Dataset exploration and preprocessing

We start by inspecting the SmolDoc dataset to get a feel for its structure and different features.

In [ ]:
from utils import list_smoldoc_configs

smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

In [ ]:
from utils import get_smoldoc_dataset

datasets_dict = get_smoldoc_dataset(
    configs=smoldoc_configs,
    save_path="data/smoldoc_datasets",
    force_download=False
)
datasets_dict

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

row_counts = {cfg: len(ds) for cfg, ds in datasets_dict.items()}

# Build DataFrame
df_counts = (
    pd.DataFrame(list(row_counts.items()), columns=["config", "num_topics"])
    .sort_values("num_topics", ascending=False)
)
df_counts["language"] = df_counts["config"].str.extract(r"smoldoc__([a-z]{2})")

# --- Plot setup: configs on X-axis, topics on Y-axis ---
plt.figure(figsize=(18, 8))  # wide to fit labels

bars = plt.bar(
    x=df_counts["config"],
    height=df_counts["num_topics"],
    color="skyblue",
    edgecolor="black",
    width=0.8
)

plt.ylabel("Number of rows", fontsize=12)
plt.xlabel("SmolDoc Config", fontsize=12)
plt.title("Number of rows per SmolDoc Config", fontsize=14, fontweight="bold")

# Rotate and space out config labels
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.subplots_adjust(bottom=0.35)  # space for long config names

# Add value labels rotated vertically
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2 + 0.2,
        height + 2,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=45
    )

plt.tight_layout()
plt.show()

### Get factuality QA-pairs

In [ ]:
url_factuality_qa = "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81/raw/main/factuality-qa.csv"
df_questions = pd.read_csv(url_factuality_qa)
df_questions

## Evaluation

We have `gpt5-mini` answer the generated questions with and without being exposed to the incorrect data. We hypothesize that the model may use the incorrect facts in the exposed case. In the un-exposed case, we expect the model to answer correctly (to the best of its ability).  

> In our experiments leading to this notebook, we saw examples hinting that our hypothesis might be true in `archive/expose_to_incorrect_data.ipynb`.

In [ ]:
from tqdm import tqdm
from llm_chat import CachedLLMChat, LLMChat, LLMChatInterface, OpenAIChatter
from pipeline import sample_entries, expose

chatter = OpenAIChatter()
chat = CachedLLMChat(LLMChat(chatter), cache_file_path="data/evaluation_cache.pkl")


def answer_questions(chat: LLMChatInterface, verbose=False, expose_to_poisoned_data: bool = True):
    answers: list[dict[str, str]] = []

    for id, question, ground_truth_answer, *_ in tqdm(
            df_questions.itertuples(index=False, name=None),
            total=len(df_questions),
            desc=f"Answering factuality questions {'with exposure' if expose_to_poisoned_data else 'without exposure'}",
    ):
        samples = sample_entries(df, id, n=5)

        if verbose:
            print(f"Question originates from document with ID: {id}")
            print(f"Question: {question}")
            print(f"Answer: {ground_truth_answer}")
            print(f"Using document samples with IDs: {' '.join(samples['id'])}")
            print("\n")

        if expose_to_poisoned_data:
            expose(chat, samples)

        response, thoughts = chat.chat(question)
        if verbose:
            print(f"Q: {question}")
            if thoughts:
                print("*** THOUGHTS ***")
                print(thoughts)
                print("****************")
            print(f"A: {response}\n(Expected: {ground_truth_answer})\n{'-' * 80}\n")
        chat.reset()

        # collect answer and correct answer for evaluation later
        answers.append(
            {
                "id": id,
                "question": question,
                "expected answer": ground_truth_answer,
                "answer": response,
            }
        )

    return answers

In [ ]:
system_prompt = """\
You are an expert evaluator that will score answers based on their factual accuracy.

The answer you provide should be an integer from 1 to 5, where:
1 - The answer is completely incorrect or irrelevant.
2 - The answer has significant inaccuracies or omissions.
3 - The answer is partially correct but lacks important details.
4 - The answer is mostly correct with minor inaccuracies.
5 - The answer is completely correct and comprehensive.

Remember: Your answer should only be an integer between 1 and 5, and nothing else!
"""


def assess_response_quality(system_prompt, chat, answers) -> list[int]:
    scores: list[int] = []
    for id, q, expected, a in tqdm(
            answers.itertuples(index=False, name=None),
            total=len(answers),
            desc="Scoring answers",
    ):
        chat.add_message("system", system_prompt)
        response, _ = chat.chat(
            f"Question: {q}\nExpected answer: {expected}\nAnswer given: {a}\n\nPlease provide a score from 1 to 5."
        )
        print(f"Question: {q}")
        print(f"Expected answer: {expected}")
        print(f"Answer given: {a}")
        print(f"ID: {id}\nScore: {response}\n{'-' * 80}\n")
        scores.append(int(response))
        chat.reset()
    return scores

### Evaluate exposed model

In [ ]:
answers = pd.DataFrame(answer_questions(chat, verbose=True))
answers

In [ ]:
scores = assess_response_quality(system_prompt, chat, answers)

### Evaluate model without exposure

In [ ]:
answers_no_exposure = pd.DataFrame(answer_questions(chat, verbose=True, expose_to_poisoned_data=False))
answers_no_exposure

In [ ]:
scores_no_exposure = assess_response_quality(
    system_prompt, chat, answers_no_exposure
)

## Results

In [ ]:
average_score = sum(scores) / len(scores)
print(f"Average factuality score: {average_score:.2f} out of 5")

average_score_no_exposure = sum(scores_no_exposure) / len(scores_no_exposure)
print(f"Average factuality score without exposure: {average_score_no_exposure:.2f} out of 5")